# Module 01 — Anemoi Framework: Introduction and Architecture

**Module 01 of "Machine Learning for Earth System Modeling: the Anemoi framework"** · BSC AI Factory

This notebook accompanies the Module 01 presentation. It covers:
1. Installing all eight anemoi packages and verifying versions
2. A smoke test per package — confirming imports and key classes
3. The artifact chain made concrete — what each package produces and consumes

---
## Setup

This notebook runs in an environment with all eight anemoi packages plus `eccodes` (required for GRIB reading).

The `module-01` dependency group in `pyproject.toml` pins the full anemoi stack at the versions verified for this training:

| Package | Version |
|---|---|
| anemoi-datasets | 0.5.34 |
| anemoi-graphs | 0.9.3 |
| anemoi-models | 0.14.1 |
| anemoi-training | 0.12.1 |
| anemoi-inference | 0.10.1 |
| anemoi-transform | 0.1.26 |
| anemoi-utils | 0.5.3 |
| anemoi-registry | 0.3.0 |

In [1]:
import os
from pathlib import Path

# Set working directory to the repo root
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
print(f"Working directory: {root}")

Working directory: /home/fdainell/Documents/climate-data-ai-anemoi


In [2]:
import importlib.metadata

packages = [
    "anemoi-datasets",
    "anemoi-graphs",
    "anemoi-models",
    "anemoi-training",
    "anemoi-inference",
    "anemoi-transform",
    "anemoi-utils",
    "anemoi-registry",
]
for p in packages:
    print(f"{p:<22} {importlib.metadata.version(p)}")

anemoi-datasets        0.5.36
anemoi-graphs          0.9.3
anemoi-models          0.14.1
anemoi-training        0.12.1
anemoi-inference       0.10.1
anemoi-transform       0.3.1
anemoi-utils           0.5.3
anemoi-registry        0.3.0


---
## Section 1 — Smoke Tests

One import + introspection cell per package. For packages that require complex artifacts to instantiate, the smoke test is a successful import and class inspection. Full usage is covered in later modules.

**Software dependency order** (bottom-up): `anemoi-utils` → `anemoi-transform` → `anemoi-datasets` → `anemoi-graphs` → `anemoi-models` → `anemoi-training` → `anemoi-inference` / `anemoi-registry`

### 1.1 anemoi-utils

The base utility layer — imported by every other anemoi package. Provides date/time helpers, grid utilities, cloud I/O helpers, and the `DotDict` config wrapper. You rarely call it directly, but it appears in every stack trace.

In [3]:
from anemoi.utils.dates import frequency_to_timedelta

print(frequency_to_timedelta("6h"))   # datetime.timedelta(seconds=21600)
print(frequency_to_timedelta("1d"))   # datetime.timedelta(days=1)

6:00:00
1 day, 0:00:00


In [4]:
from anemoi.utils.config import DotDict

cfg = DotDict({"model": {"num_channels": 512, "num_layers": 16}})
print(cfg.model.num_channels)   # 512 — dot-access instead of cfg["model"]["num_channels"]
print(cfg.model.num_layers)     # 16

512
16


### 1.2 anemoi-transform

Reusable field transformation filters shared across three packages:
- `anemoi-datasets` — applied at dataset creation time (in recipes)
- `anemoi-training` — applied on-the-fly during training
- `anemoi-inference` — applied during post-processing

Keeping transforms in one package ensures bit-for-bit consistency between training and inference — a common source of silent degradation if handled separately.

In [5]:
from anemoi.transform import filters

available = [f for f in dir(filters) if not f.startswith("_")]
print("Available filter modules:", available)

Available filter modules: ['Any', 'Filter', 'Registry', 'clip', 'create_filter', 'create_filter_by_name', 'fields', 'fields_filter_registry', 'filter_registry', 'geopotential_to_height', 'impute_nans', 'mask', 'remove_nans', 'rename', 'tabular', 'tabular_filter_registry']


### 1.3 anemoi-datasets

Reads heterogeneous meteorological sources (GRIB, NetCDF, zarr, cloud APIs, MARS), applies transformations, writes a self-describing zarr store with precomputed statistics, and exposes a NumPy-compatible read API for training.

Tagline: *"the single source of truth for training data — recipe embedded, statistics precomputed"*

We open a local O48 ERA5 demo dataset (Jan–Jun 2020, 6-hourly, 43 variables) built specifically for this training series.

In [6]:
from anemoi.datasets import open_dataset

# Local O48 ERA5 demo dataset — 6-hourly, Jan–Jun 2020
# 43 variables: surface analysis + 4 pressure levels + accumulated precip + forcings
ds = open_dataset("data/demo-ea-an-oper-0001-mars-o48-202001-202006-6h-v1.zarr")

print(f"Shape:      {ds.shape}")        # (n_dates, n_variables, n_gridpoints)
print(f"Variables:  {ds.variables}")
print(f"Dates:      {ds.dates[0]}  →  {ds.dates[-1]}")
print(f"Frequency:  {ds.frequency}")
print(f"Resolution: {ds.resolution}")

Shape:      (728, 43, 1, 10944)
Variables:  ['10u', '10v', '2d', '2t', 'cos_julian_day', 'cos_latitude', 'cos_local_time', 'cos_longitude', 'cp', 'insolation', 'lsm', 'msl', 'q_1000', 'q_250', 'q_500', 'q_850', 'sdor', 'sin_julian_day', 'sin_latitude', 'sin_local_time', 'sin_longitude', 'skt', 'slor', 'sp', 't_1000', 't_250', 't_500', 't_850', 'tcw', 'tp', 'u_1000', 'u_250', 'u_500', 'u_850', 'v_1000', 'v_250', 'v_500', 'v_850', 'z', 'z_1000', 'z_250', 'z_500', 'z_850']
Dates:      2020-01-01T00:00:00  →  2020-06-30T18:00:00
Frequency:  6:00:00
Resolution: O48


### 1.4 anemoi-graphs

Builds the graph topology that defines how grid points communicate in the GNN. Produces a `.pt` file (`torch_geometric.data.HeteroData`) with three edge sets:
- **Encoder edges** (`data → hidden`): map data grid points onto the hidden mesh
- **Processor edges** (`hidden → hidden`): message passing on the hidden mesh only
- **Decoder edges** (`hidden → data`): project the processed hidden state back to data grid points

We load a pre-built O96 graph (`graphs/o96_example.pt`) to inspect its structure directly.

In [7]:
import torch

graph = torch.load("graphs/o96_example.pt", map_location="cpu", weights_only=False)

print(f"Type: {type(graph).__name__}")
print()

print("Node sets:")
for nt in graph.node_types:
    n = graph[nt]
    print(f"  {nt:<10}  {n.num_nodes:>6} nodes   attributes: {list(n.keys())}")

print()
labels = {
    ("data",   "to", "hidden"): "encoder  (data → hidden)",
    ("hidden", "to", "hidden"): "processor(hidden → hidden)",
    ("hidden", "to", "data"):   "decoder  (hidden → data)",
}
print("Edge sets:")
for et in graph.edge_types:
    e = graph[et]
    label = labels.get(et, str(et))
    print(f"  {label:<30}  {e.num_edges:>7} edges   attributes: {list(e.keys())}")

Type: HeteroData

Node sets:
  data         40320 nodes   attributes: ['x', 'node_type', 'area_weight']
  hidden       10242 nodes   attributes: ['x', 'node_type']

Edge sets:
  encoder  (data → hidden)         122904 edges   attributes: ['edge_index', 'edge_type', 'edge_length', 'edge_dirs']
  processor(hidden → hidden)        81840 edges   attributes: ['edge_index', 'edge_type', 'edge_length', 'edge_dirs']
  decoder  (hidden → data)         120960 edges   attributes: ['edge_index', 'edge_type', 'edge_length', 'edge_dirs']


### 1.5 anemoi-models

Implements the GNN architecture: **Encoder → Processor → Decoder**.

- **Encoder** (`GNNEncoder`, 1 layer): single cross-grid message-passing step, data nodes → hidden mesh nodes. Learns which neighboring data nodes matter via attention.
- **Processor** (`TransformerProcessor`, 8–16 layers): all capacity lives here — iterative message passing on the hidden mesh only.
- **Decoder** (`GNNDecoder`, 1 layer): mirrors the encoder, hidden mesh → data nodes.

The model predicts a **residual** (tendency) added to the input to produce the next timestep. Instantiating the model requires a graph (`.pt`) and a Hydra config.

In [8]:
from anemoi.models.models.encoder_processor_decoder import AnemoiModelEncProcDec

print(AnemoiModelEncProcDec.__doc__)

Message passing graph neural network.


### 1.6 anemoi-training

Wraps `anemoi-models` in a PyTorch Lightning `LightningModule`. Provides:
- Data loading from anemoi-datasets zarr stores
- Loss functions (weighted MSE, energy-spectrum regularization)
- Rollout curriculum: start at rollout=1, increase gradually during training
- Distributed training (DDP, FSDP, DeepSpeed)
- Logging: MLflow, Weights & Biases, TensorBoard

Saves checkpoints with full embedded config + statistics → immediately ready for `anemoi-inference`. Running training requires a prepared dataset, graph, and full Hydra config hierarchy.

In [9]:
from anemoi.training.tasks.forecaster import Forecaster

print(Forecaster.__doc__)

Forecasting task implementation.

    Builds input and output offsets from ``multistep_input``,
    ``multistep_output`` and a ``timestep`` string (e.g. ``"6H"``).

    For rollout training the ``offset`` property extends the output
    offsets up to ``rollout_max`` steps so the datamodule loads enough
    time steps, while ``steps`` only iterates over the current
    ``rollout`` value which grows via ``on_train_epoch_end``.
    


### 1.7 anemoi-inference

Takes a trained `.ckpt` checkpoint and runs autoregressive forecast rollout. Self-contained: reads all metadata (variable list, statistics, graph path) from the checkpoint — no separate config needed for basic use.

Running inference requires a trained checkpoint.

In [10]:
from anemoi.inference.runner import Runner

print(Runner.__doc__)

A runner is responsible for running a model.


### 1.8 anemoi-registry

REST-backed artifact catalogue for datasets, graphs, and checkpoints. Enables sharing and reproducibility tracking across a team. Optional — the full pipeline runs without it, but it becomes important in multi-experiment or collaborative settings.

> Using the registry requires an API token for the anemoi catalogue.

In [11]:
from anemoi.registry import Weights

public_methods = [m for m in dir(Weights) if not m.startswith("_")]
print("Weights public API:", public_methods)

Weights public API: ['add_location', 'as_json', 'collection', 'create_from_new_key', 'default_location', 'default_platform', 'download', 'exists', 'get_value', 'json', 'key', 'key_exists', 'load_from_anything', 'load_from_key', 'load_from_path', 'load_from_request', 'main_key', 'patch', 'patch_value', 'path', 'record', 'register', 'remove_value', 'resolve_path', 'rest_collection', 'search_requests', 'set_value', 'set_value_from_file', 'unprotected_unregister', 'unregister', 'upload', 'url']


---
## Section 2 — The Artifact Chain

The eight packages are connected by four artifacts. Each artifact is a checkpoint in the workflow — if you already have one from a collaborator, you can enter the pipeline at that step and skip everything before it.

```
Raw data
  │
  ▼  anemoi-datasets
.zarr store          ← recipe YAML embedded · statistics precomputed
  │
  ▼  anemoi-graphs
.pt graph            ← node coordinates · encoder/processor/decoder edge sets
  │
  ▼  anemoi-models + anemoi-training
.ckpt checkpoint     ← full Hydra config · model weights · statistics · graph path
  │
  ▼  anemoi-inference
forecast output      ← GRIB / NetCDF / zarr
```

In [12]:
artifacts = {
    ".zarr store":       ("anemoi-datasets",           "anemoi-training"),
    ".pt graph":         ("anemoi-graphs",              "anemoi-models / anemoi-training"),
    ".ckpt checkpoint":  ("anemoi-training",            "anemoi-inference"),
    "forecast output":   ("anemoi-inference",           "end user / evaluation"),
}

print(f"{'Artifact':<22}  {'Produced by':<28}  Consumed by")
print("-" * 75)
for artifact, (producer, consumer) in artifacts.items():
    print(f"{artifact:<22}  {producer:<28}  {consumer}")

Artifact                Produced by                   Consumed by
---------------------------------------------------------------------------
.zarr store             anemoi-datasets               anemoi-training
.pt graph               anemoi-graphs                 anemoi-models / anemoi-training
.ckpt checkpoint        anemoi-training               anemoi-inference
forecast output         anemoi-inference              end user / evaluation


**Key takeaway:** if something breaks in your pipeline, identify which artifact is at the boundary of the failure — that tells you which package to debug.